# 🧪 Notebook 4: Evaluación del Agente

## Objetivos
- Diseñar test cases para evaluar un agente de IA
- Crear un dataset de evaluación en Langfuse
- Ejecutar evaluaciones programáticas (determinísticas + LLM-as-judge)
- Comparar prompt v1 vs v2 con datos objetivos
- Entender cuándo usar evaluación offline vs online

## ¿Por qué evaluar?
> "Sin evaluación, estás optimizando vibes." — Hamel Husain

No basta con probar 3 queries manualmente. Necesitamos:
- **Cobertura**: probar edge cases que no se nos ocurren en el momento
- **Regresión**: detectar si un cambio rompe algo que antes funcionaba
- **Objetividad**: métricas reproducibles, no "me parece que funciona"

## 1. Setup

In [ ]:
# Verificar dependencias (instaladas via: cd notebooks/ && uv sync)
import strands, langfuse, boto3
print(f"✅ strands {strands.__version__}")
print(f"✅ langfuse {langfuse.__version__}")
print(f"✅ boto3 {boto3.__version__}")

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

from langfuse import Langfuse, observe

langfuse = Langfuse()
langfuse.auth_check()
print("✅ Conectado a Langfuse")

## 2. Diseñar test cases

Un buen dataset de evaluación tiene:
- **Casos positivos**: queries que el agente DEBE responder correctamente
- **Casos negativos**: queries que el agente DEBE rechazar
- **Edge cases**: queries ambiguas, mal escritas, fuera de ámbito

### Tipos de evaluación

| Tipo | Cómo funciona | Cuándo usar |
|------|--------------|------------|
| **Determinística** | Comparar string, regex, contiene | Respuestas exactas (precios, nombres) |
| **LLM-as-judge** | Otro LLM evalúa la respuesta | Calidad, relevancia, fidelidad |
| **Humana** | Una persona revisa | Gold standard, casos complejos |

In [ ]:
# Dataset de evaluación para TechShop
TEST_CASES = [
    # --- Productos (el agente DEBE usar search_catalog) ---
    {
        "input": "¿Qué portátiles tenéis?",
        "category": "product",
        "expected_tools": ["search_catalog"],
        "expected_contains": ["ProBook", "NanoBook"],
        "should_answer": True,
    },
    {
        "input": "¿Cuánto cuesta el VoltPhone S?",
        "category": "product",
        "expected_tools": ["search_catalog"],
        "expected_contains": ["749"],
        "should_answer": True,
    },
    {
        "input": "¿Tenéis auriculares con cancelación de ruido?",
        "category": "product",
        "expected_tools": ["search_catalog"],
        "expected_contains": ["AuralPods Pro"],
        "should_answer": True,
    },
    # --- FAQs (el agente DEBE usar get_faq_answer) ---
    {
        "input": "¿Cuál es la política de devoluciones?",
        "category": "faq",
        "expected_tools": ["get_faq_answer"],
        "expected_contains": ["30 días"],
        "should_answer": True,
    },
    {
        "input": "¿Hacéis envíos internacionales?",
        "category": "faq",
        "expected_tools": ["get_faq_answer"],
        "expected_contains": ["no realizamos"],
        "should_answer": True,
    },
    {
        "input": "¿Puedo pagar en cuotas?",
        "category": "faq",
        "expected_tools": ["get_faq_answer"],
        "expected_contains": ["cuotas"],
        "should_answer": True,
    },
    # --- Fuera de ámbito (el agente NO DEBE responder) ---
    {
        "input": "¿Cuál es la capital de Francia?",
        "category": "out_of_scope",
        "expected_tools": [],
        "expected_contains": [],
        "should_answer": False,
    },
    {
        "input": "Escríbeme un poema sobre el amor",
        "category": "out_of_scope",
        "expected_tools": [],
        "expected_contains": [],
        "should_answer": False,
    },
    # --- Edge cases ---
    {
        "input": "quiero algo pa escuchar musica corriendo",
        "category": "product_ambiguous",
        "expected_tools": ["search_catalog"],
        "expected_contains": [],
        "should_answer": True,
    },
    {
        "input": "¿Tenéis iPhone?",
        "category": "product_not_found",
        "expected_tools": ["search_catalog"],
        "expected_contains": [],
        "should_answer": True,  # Debe responder que no lo tiene
    },
]

print(f"📋 Test cases definidos: {len(TEST_CASES)}")
for tc in TEST_CASES:
    print(f"   [{tc['category']:20s}] {tc['input']}")

## 3. Crear dataset en Langfuse

Subimos los test cases a Langfuse como un **Dataset** para poder reutilizarlos.

In [ ]:
DATASET_NAME = "techshop-eval-v1"

# Crear dataset
try:
    dataset = langfuse.create_dataset(name=DATASET_NAME)
    print(f"✅ Dataset '{DATASET_NAME}' creado")
except Exception:
    print(f"ℹ️  Dataset '{DATASET_NAME}' ya existe")

# Subir test cases como items
for i, tc in enumerate(TEST_CASES):
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        input={"query": tc["input"]},
        expected_output={
            "category": tc["category"],
            "expected_contains": tc["expected_contains"],
            "should_answer": tc["should_answer"],
        },
        metadata={"expected_tools": tc["expected_tools"]},
    )

langfuse.flush()
print(f"✅ {len(TEST_CASES)} test cases subidos a Langfuse")

## 4. Preparar el agente para evaluación

In [ ]:
from techshop_agent import create_agent

print("✅ Agente preparado para evaluación")

## 5. Ejecutar evaluación determinística

La evaluación determinística comprueba cosas que podemos verificar automáticamente:
- ¿La respuesta contiene las keywords esperadas?
- ¿El agente rechazó queries fuera de ámbito?

In [ ]:
# Prompts v1 y v2
PROMPT_V1 = """Eres un asistente de atención al cliente para TechShop.
SIEMPRE usa las herramientas disponibles para buscar información antes de responder.
Responde en español, de forma concisa y profesional.
Si no encuentras la información, dilo honestamente.
"""

PROMPT_V2 = """Eres un asistente de atención al cliente para TechShop, una tienda online de electrónica.

## Tu ámbito
SOLO puedes ayudar con:
- Consultas sobre productos del catálogo de TechShop
- Preguntas frecuentes de TechShop (envíos, devoluciones, garantías, pagos, horarios)

## Reglas estrictas
1. SIEMPRE usa las herramientas disponibles ANTES de responder
2. SOLO responde con información que las herramientas devuelvan
3. Si las herramientas no devuelven resultados, di: "No he encontrado esa información"
4. Si la consulta NO es sobre TechShop, responde: "Solo puedo ayudarte con consultas sobre TechShop."
5. NUNCA inventes precios, stock, o políticas
"""

def run_evaluation(prompt: str, prompt_version: str):
    """Ejecuta todos los test cases contra un prompt específico."""
    agent = create_agent(system_prompt=prompt)
    results = []

    for tc in TEST_CASES:
        query = tc["input"]
        response = str(agent(query))

        # Evaluaciones determinísticas
        contains_check = all(
            keyword.lower() in response.lower()
            for keyword in tc["expected_contains"]
        ) if tc["expected_contains"] else True

        # Para out_of_scope: verificar que rechaza
        if not tc["should_answer"]:
            rejection_keywords = ["no puedo", "solo puedo", "fuera", "no es mi ámbito", "lo siento"]
            scope_check = any(k in response.lower() for k in rejection_keywords)
        else:
            scope_check = True

        passed = contains_check and scope_check
        results.append({
            "query": query,
            "category": tc["category"],
            "response_preview": response[:100],
            "contains_check": contains_check,
            "scope_check": scope_check,
            "passed": passed,
            "prompt_version": prompt_version,
        })

        status = "✅" if passed else "❌"
        print(f"  {status} [{tc['category']:20s}] {query[:40]}")

    return results

print("🧪 Evaluación con prompt v1:")
results_v1 = run_evaluation(PROMPT_V1, "v1")

print(f"\n🧪 Evaluación con prompt v2:")
results_v2 = run_evaluation(PROMPT_V2, "v2")

langfuse.flush()

## 6. Comparar resultados v1 vs v2

In [ ]:
def summarize_results(results, version):
    total = len(results)
    passed = sum(1 for r in results if r["passed"])
    by_category = {}
    for r in results:
        cat = r["category"]
        if cat not in by_category:
            by_category[cat] = {"total": 0, "passed": 0}
        by_category[cat]["total"] += 1
        by_category[cat]["passed"] += 1 if r["passed"] else 0
    return {"version": version, "total": total, "passed": passed, "rate": passed/total, "by_category": by_category}

s1 = summarize_results(results_v1, "v1")
s2 = summarize_results(results_v2, "v2")

print("📊 Resultados comparados:\n")
print(f"{'Métrica':<30s} {'v1':>10s} {'v2':>10s}")
print("-" * 52)
print(f"{'Pass rate total':<30s} {s1['rate']:>9.0%} {s2['rate']:>9.0%}")
print()
for cat in sorted(set(list(s1["by_category"].keys()) + list(s2["by_category"].keys()))):
    c1 = s1["by_category"].get(cat, {"passed": 0, "total": 0})
    c2 = s2["by_category"].get(cat, {"passed": 0, "total": 0})
    r1 = f"{c1['passed']}/{c1['total']}" if c1["total"] > 0 else "N/A"
    r2 = f"{c2['passed']}/{c2['total']}" if c2["total"] > 0 else "N/A"
    print(f"  {cat:<28s} {r1:>10s} {r2:>10s}")

## 7. Evaluación con LLM-as-judge

Para evaluar **calidad** (no solo si contiene una keyword), usamos otro LLM como juez.
El juez recibe: la consulta, la respuesta del agente, y un criterio de evaluación.

> **Concepto:** LLM-as-judge es costoso pero captura dimensiones imposibles de evaluar con regex: coherencia, tono, fidelidad, relevancia.

In [ ]:
import boto3

bedrock_runtime = boto3.client("bedrock-runtime", region_name=os.getenv("AWS_REGION", "us-east-1"))

def llm_judge(query: str, response: str, criteria: str) -> dict:
    """Usa un LLM como juez para evaluar la calidad de una respuesta.

    Returns:
        dict con score (1-5) y justificación
    """
    judge_prompt = f"""Evalúa la siguiente respuesta de un agente de customer service.

Consulta del usuario: {query}
Respuesta del agente: {response}

Criterio de evaluación: {criteria}

Responde SOLO con JSON válido:
{{"score": <1-5>, "justification": "<explicación breve>"}}

Donde 1 = muy malo, 5 = excelente.
"""

    result = bedrock_runtime.converse(
        modelId=os.getenv("MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0"),
        messages=[{"role": "user", "content": [{"text": judge_prompt}]}],
        inferenceConfig={"maxTokens": 200, "temperature": 0.0},
    )

    response_text = result["output"]["message"]["content"][0]["text"]
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        return {"score": 0, "justification": f"Error parsing: {response_text[:100]}"}

# Evaluar algunas respuestas con LLM-as-judge
print("🤖 Evaluación LLM-as-judge (usando Bedrock):\n")

judge_criteria = {
    "relevancia": "¿La respuesta es relevante a la consulta del usuario?",
    "fidelidad": "¿La respuesta se basa solo en datos reales o inventa información?",
    "profesionalidad": "¿El tono es profesional y apropiado para customer service?",
}

# Evaluar 3 test cases representativos
sample_cases = TEST_CASES[:3]
agent_v2 = create_agent(PROMPT_V2)

for tc in sample_cases:
    query = tc["input"]
    response = str(agent_v2(query))

    print(f"\nQuery: {query}")
    print(f"Response: {response[:100]}...")

    for criteria_name, criteria_text in judge_criteria.items():
        result = llm_judge(query, response, criteria_text)
        print(f"  {criteria_name}: {result['score']}/5 — {result['justification']}")

langfuse.flush()

## 8. Concepto: Evaluación en CI/CD

### El pipeline ideal
```
Desarrollador cambia prompt
    │
    ├─→ PR en GitHub
    │
    ├─→ GitHub Actions ejecuta evaluación
    │   ├── Evaluaciones determinísticas (rápidas, gratis)
    │   └── LLM-as-judge (lentas, cuestan tokens)
    │
    ├─→ Si pass rate > threshold → ✅ Merge permitido
    └─→ Si pass rate < threshold → ❌ Merge bloqueado
```

### Herramientas para CI/CD de LLMs
| Herramienta | Tipo | Ventaja |
|-------------|------|--------|
| **promptfoo** | CLI (Node.js) | Config YAML, muchos providers, comparación visual |
| **deepeval** | Python lib | Pytest-friendly, métricas predefinidas |
| **Langfuse Datasets** | API | Integrado con trazas, sin dependencias extra |
| **Custom Python** | Scripts | Máximo control, sin dependencias |

> **Para este curso:** Usamos Langfuse Datasets + Python custom. En producción, añadirías promptfoo o deepeval según preferencia.

## Resumen

| Concepto | Qué aprendimos |
|----------|----------------|
| **Test cases** | Diseñar evaluaciones: positivos, negativos, edge cases |
| **Datasets** | Almacenar test cases en Langfuse para reutilización |
| **Eval determinística** | Verificar keywords, regex, scope — rápido y gratis |
| **LLM-as-judge** | Evaluar calidad con otro LLM — potente pero costoso |
| **v1 vs v2** | Comparar versiones con datos objetivos |
| **CI/CD** | Automatizar evaluaciones como gate de deploy |

> **Concepto clave:** Del "deployamos y rezamos" al "deployamos cuando pasa los tests".

---

## Siguiente: [Notebook 5 - Guardrails con Bedrock](../day_3/05_guardrails.ipynb)